In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import math
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
uploaded = files.upload()

Saving meal_info.csv to meal_info.csv


In [ ]:
uploaded = files.upload()

Saving fulfilment_center_info.csv to fulfilment_center_info.csv


In [ ]:
train = pd.read_csv("train.csv")
meal_info = pd.read_csv("meal_info.csv")
center_info = pd.read_csv("fulfilment_center_info.csv")

train.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177
1,1466964,1,55,1993,136.83,135.83,0,0,270
2,1346989,1,55,2539,134.86,135.86,0,0,189
3,1338232,1,55,2139,339.50,437.53,0,0,54
4,1448490,1,55,2631,243.50,242.50,0,0,40


In [ ]:

df = train.merge(meal_info, on="meal_id", how="left")


df = df.merge(center_info, on="center_id", how="left")

print("Columns after merge:")
print(df.columns)



Columns after merge:
Index(['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price',
       'emailer_for_promotion', 'homepage_featured', 'num_orders', 'category',
       'cuisine', 'city_code', 'region_code', 'center_type', 'op_area'],
      dtype='object')


In [ ]:

categorical_cols = df.select_dtypes(include=['object']).columns


df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:

df = df.sort_values(["center_id", "meal_id", "week"])


df["lag_1"] = df.groupby(["center_id", "meal_id"])["num_orders"].shift(1)
df["lag_2"] = df.groupby(["center_id", "meal_id"])["num_orders"].shift(2)
df["lag_3"] = df.groupby(["center_id", "meal_id"])["num_orders"].shift(3)
df["lag_4"] = df.groupby(["center_id", "meal_id"])["num_orders"].shift(4)


df["rolling_mean_4"] = (
    df.groupby(["center_id", "meal_id"])["num_orders"]
      .shift(1)
      .rolling(4)
      .mean()
)

df["rolling_mean_8"] = (
    df.groupby(["center_id", "meal_id"])["num_orders"]
      .shift(1)
      .rolling(8)
      .mean()
)


df["discount"] = (
    (df["base_price"] - df["checkout_price"]) / df["base_price"]
)


df = df.dropna()

print("Feature engineering complete.")
print("Remaining rows:", len(df))

Feature engineering complete.
Remaining rows: 427851


In [ ]:
target = "num_orders"

cat_features = ["category", "cuisine", "center_type"]

features = df.columns.tolist()
features.remove(target)
features.remove("id")

In [ ]:
max_week = df["week"].max()
split_week = int(max_week * 0.8)

train_df = df[df["week"] <= split_week].copy()
valid_df = df[df["week"] > split_week].copy()

X_train = train_df[features]
y_train = train_df[target]

X_valid = valid_df[features]
y_valid = valid_df[target]

print("Train weeks:", train_df["week"].min(), "-", train_df["week"].max())
print("Valid weeks:", valid_df["week"].min(), "-", valid_df["week"].max())

Train weeks: 9 - 116
Valid weeks: 117 - 145


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.metrics import r2_score

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_valid)

rf_rmse = np.sqrt(mean_squared_error(y_valid, rf_preds))

print("Random Forest RMSE:", rf_rmse)
print("R_2 score: ", r2_score(y_valid, rf_preds))

Random Forest RMSE: 162.21375742454035
R_2 score:  0.8030228154800145


In [ ]:
from catboost import CatBoostRegressor
import numpy as np
from sklearn.metrics import mean_squared_error

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    verbose=200,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid)
)

# Predictions
cb_preds = model.predict(X_valid)

cb_rmse = np.sqrt(mean_squared_error(y_valid, cb_preds))
print("CatBoost RMSE:", cb_rmse)
print("R_2 score: ", r2_score(y_valid, cb_preds))

0:	learn: 371.8032832	test: 354.0133927	best: 354.0133927 (0)	total: 100ms	remaining: 1m 40s
200:	learn: 147.3853814	test: 157.8071576	best: 157.7875414 (199)	total: 10.1s	remaining: 40s
400:	learn: 132.3640815	test: 155.5557524	best: 155.5167738 (382)	total: 19.9s	remaining: 29.7s
600:	learn: 123.1225791	test: 155.2686371	best: 154.8392571 (476)	total: 30.3s	remaining: 20.1s
800:	learn: 116.9468832	test: 155.1337014	best: 154.8392571 (476)	total: 40.6s	remaining: 10.1s
999:	learn: 112.5146025	test: 154.9198291	best: 154.8392571 (476)	total: 51.2s	remaining: 0us

bestTest = 154.8392571
bestIteration = 476

Shrink model to first 477 iterations.
CatBoost RMSE: 154.8392570911099
R_2 score:  0.8205255152669052


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

preds = model.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, preds))
r2 = r2_score(y_valid, preds)

print("Validation RMSE:", rmse)
print("Validation R2:", r2)

Validation RMSE: 154.8392570911099
Validation R2: 0.8205255152669052


In [ ]:
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

lgb_model = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=256,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=0.3,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1
)

In [ ]:
lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        early_stopping(stopping_rounds=100),
        log_evaluation(200)
    ]
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023400 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2651
[LightGBM] [Info] Number of data points in the train set: 332861, number of used features: 35
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 265.478749
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 171.171	valid_0's l2: 29299.4
[400]	valid_0's rmse: 156.059	valid_0's l2: 24354.3
[600]	valid_0's rmse: 153.948	valid_0's l2: 23700.1
[800]	valid_0's rmse: 153.747	valid_0's l2: 23638
Early stopping, best iteration is:
[701]	valid_0's rmse: 153.647	valid_0's l2: 23607.5


LGBMRegressor(colsample_bytree=0.85, learning_rate=0.01, n_estimators=3000,
              n_jobs=-1, num_leaves=256, random_state=42, reg_alpha=0.2,
              reg_lambda=0.3, subsample=0.85)

In [ ]:

lgb_preds = lgb_model.predict(X_valid)
lgb_rmse = np.sqrt(mean_squared_error(y_valid, lgb_preds))
lgb_r2 = r2_score(y_valid, lgb_preds)
print("LightGBM RMSE:", lgb_rmse)
print("LightGBM R2:", lgb_r2)

LightGBM RMSE: 153.64721375790793
LightGBM R2: 0.8232782776350696


In [ ]:
print("Target in features?", "num_orders" in X_train.columns)

Target in features? False


In [ ]:
df.columns

Index(['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price',
       'emailer_for_promotion', 'homepage_featured', 'num_orders', 'city_code',
       'region_code', 'op_area', 'category_Biryani', 'category_Desert',
       'category_Extras', 'category_Fish', 'category_Other Snacks',
       'category_Pasta', 'category_Pizza', 'category_Rice Bowl',
       'category_Salad', 'category_Sandwich', 'category_Seafood',
       'category_Soup', 'category_Starters', 'cuisine_Indian',
       'cuisine_Italian', 'cuisine_Thai', 'center_type_TYPE_B',
       'center_type_TYPE_C', 'lag_1', 'lag_2', 'lag_3', 'lag_4',
       'rolling_mean_4', 'rolling_mean_8', 'discount'],
      dtype='object')

In [ ]:
target = "num_orders"
drop_cols = ["id", target]
feature_cols = [col for col in df.columns if col not in drop_cols]
final_df = df[feature_cols + [target]].copy()

print("Final Dataset Shape:", final_df.shape)
print("Number of Features:", len(feature_cols))

Final Dataset Shape: (427851, 36)
Number of Features: 35


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

final_df.to_csv('/content/drive/MyDrive/historical_data.csv', index=False)

Mounted at /content/drive


In [ ]:
import joblib
joblib.dump(model, "lightgbm_model.pkl")
joblib.dump(feature_cols, "feature_columns.pkl")

['feature_columns.pkl']